
# Wind Intraday Unit Mismatch Check

Goal: Verify whether intraday forecast/actual series are in inconsistent units (kW vs MW).


In [40]:

import polars as pl
import pandas as pd
import plotly.express as px
from pathlib import Path

base = Path.cwd()
if base.name == "notebooks":
    base = base.parent

cand = [
    base / "data/processed/all_data_transformed.parquet",
    base / "data/processed/all_data_clean.parquet",
    base / "data/processed/all_data.parquet",
]
DATA_PATH = next((p for p in cand if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No input parquet found in data/processed/")


In [41]:

# Load data
cols = [
    "timestamp_utc",
    "wind_onshore_forecast_intraday",
    "wind_onshore_actual",
    "wind_offshore_forecast",
    "wind_offshore_actual",
    "total_wind_intraday_forecast",
]

df = pl.read_parquet(DATA_PATH, columns=[c for c in cols if c in pl.read_parquet(DATA_PATH).columns])
print("rows", df.height, "cols", df.columns)


rows 44571 cols ['timestamp_utc', 'wind_onshore_forecast_intraday', 'wind_onshore_actual', 'wind_offshore_forecast', 'wind_offshore_actual', 'total_wind_intraday_forecast']


In [42]:

# Basic scale comparison
for c in ["wind_onshore_forecast_intraday", "wind_onshore_actual", "wind_offshore_forecast", "wind_offshore_actual", "total_wind_intraday_forecast"]:
    if c in df.columns:
        stats = df.select([
            pl.col(c).min().alias("min"),
            pl.col(c).median().alias("median"),
            pl.col(c).mean().alias("mean"),
            pl.col(c).max().alias("max"),
        ]).to_dicts()[0]
        print(c, stats)


wind_onshore_forecast_intraday {'min': 301.3125, 'median': 7537.0, 'mean': 7902.305449772255, 'max': 39283.0}
wind_onshore_actual {'min': 46.5, 'median': 9263.25, 'mean': 12066.795134844066, 'max': 48499.5}
wind_offshore_forecast {'min': 13.0, 'median': 2634.75, 'mean': 2830.6299490688803, 'max': 6834.5}
wind_offshore_actual {'min': 0.0, 'median': 2601.75, 'mean': 2842.1639308952213, 'max': 7941.53}
total_wind_intraday_forecast {'min': 1163.5, 'median': 10448.7475, 'mean': 10732.815193977605, 'max': 43229.5}


In [43]:

# Ratio check: forecast vs actual
if {"wind_onshore_forecast_intraday", "wind_onshore_actual"}.issubset(df.columns):
    ratio = (
        df.select([
            (pl.col("wind_onshore_forecast_intraday") / pl.col("wind_onshore_actual")).alias("ratio")
        ])
        .drop_nulls()
    )
    print("ratio stats:", ratio.select([
        pl.col("ratio").quantile(0.05).alias("p05"),
        pl.col("ratio").median().alias("p50"),
        pl.col("ratio").quantile(0.95).alias("p95"),
    ]).to_dicts()[0])


ratio stats: {'p05': 0.1739119490035397, 'p50': 0.8599775419166282, 'p95': 5.869839857651246}


In [44]:

# Scatter: intraday forecast vs actual (sample)
if {"wind_onshore_forecast_intraday", "wind_onshore_actual"}.issubset(df.columns):
    sample = df.select(["wind_onshore_forecast_intraday", "wind_onshore_actual"]).drop_nulls().sample(n=min(20000, df.height))
    fig = px.scatter(sample.to_pandas(), x="wind_onshore_actual", y="wind_onshore_forecast_intraday",
                     opacity=0.2, title="Intraday Forecast vs Actual (Onshore)")
    fig.show()


In [45]:

# Hypothesis test: rescale by 1000 and re-check ratios
if {"wind_onshore_forecast_intraday", "wind_onshore_actual"}.issubset(df.columns):
    ratio_scaled = (
        df.select([
            ((pl.col("wind_onshore_forecast_intraday") / 1000) / pl.col("wind_onshore_actual")).alias("ratio")
        ])
        .drop_nulls()
    )
    print("ratio stats after /1000:", ratio_scaled.select([
        pl.col("ratio").quantile(0.05).alias("p05"),
        pl.col("ratio").median().alias("p50"),
        pl.col("ratio").quantile(0.95).alias("p95"),
    ]).to_dicts()[0])


ratio stats after /1000: {'p05': 0.00017391194900353972, 'p50': 0.0008599775419166282, 'p95': 0.005869839857651246}



## Forensic Investigation: Wind Onshore Intraday ~3x Mismatch


In [46]:

import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

base = Path.cwd()
if base.name == "notebooks":
    base = base.parent

cand = [
    base / "data/processed/all_data_transformed.parquet",
    base / "data/processed/all_data_clean.parquet",
    base / "data/processed/all_data.parquet",
]
DATA_PATH = next((p for p in cand if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No input parquet found in data/processed/")

_df = pl.read_parquet(DATA_PATH)


In [47]:

# Step 1: DA vs Intraday baseline ratio
req = ["wind_onshore_forecast", "wind_onshore_forecast_intraday", "wind_onshore_actual"]
missing = [c for c in req if c not in _df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

baseline = (
    _df.select(req)
       .filter(pl.col("wind_onshore_actual") > 0)
       .drop_nulls()
       .with_columns(
           (pl.col("wind_onshore_forecast_intraday") / pl.col("wind_onshore_forecast")).alias("da_to_id_ratio"),
           (pl.col("wind_onshore_forecast") - pl.col("wind_onshore_actual")).abs().alias("da_abs_err"),
           (pl.col("wind_onshore_forecast_intraday") - pl.col("wind_onshore_actual")).abs().alias("id_abs_err"),
       )
)

summary = baseline.select([
    pl.col("da_abs_err").mean().alias("DA_MAE"),
    pl.col("id_abs_err").mean().alias("ID_MAE"),
    pl.col("da_to_id_ratio").median().alias("DA_to_ID_Ratio")
]).to_pandas()
summary


,DA_MAE,ID_MAE,DA_to_ID_Ratio
0,1102.923208,8187.504576,0.883797


In [48]:

# Step 2: Ratio stability (CV)
ratio_df = (
    _df.select(["wind_onshore_forecast_intraday", "wind_onshore_actual"])
       .filter(pl.col("wind_onshore_actual") > 0)
       .drop_nulls()
       .with_columns(
           (pl.col("wind_onshore_forecast_intraday") / pl.col("wind_onshore_actual")).alias("ratio")
       )
)

ratio_stats = ratio_df.select([
    pl.col("ratio").mean().alias("mean"),
    pl.col("ratio").std().alias("std"),
    (pl.col("ratio").std() / pl.col("ratio").mean()).alias("cv"),
]).to_pandas()
ratio_stats


,mean,std,cv
0,1.733019,3.881314,2.239626


In [49]:

# Step 3: Correlation (Pearson + Spearman)
from scipy.stats import spearmanr

corr_df = ratio_df.select(["wind_onshore_forecast_intraday", "wind_onshore_actual"]).to_pandas()
pearson = corr_df.corr(method="pearson").iloc[0,1]
spearman = spearmanr(corr_df["wind_onshore_forecast_intraday"], corr_df["wind_onshore_actual"]).correlation

print("Pearson:", pearson)
print("Spearman:", spearman)


Pearson: -0.1491601489902184
Spearman: -0.2592370050737058


In [50]:

# Step 4: Visualization
# 1-week sample (random window)
if "timestamp_utc" in _df.columns:
    sample = (
        _df.select(["timestamp_utc", "wind_onshore_actual", "wind_onshore_forecast", "wind_onshore_forecast_intraday"])
           .drop_nulls()
    )
    if sample.height > 0:
        ts_min = sample.select(pl.col("timestamp_utc").min()).item()
        ts_max = sample.select(pl.col("timestamp_utc").max()).item()
        rng = np.random.default_rng(42)
        start = ts_min + (ts_max - ts_min) * rng.random()
        end = start + pl.duration(days=7)
        window = sample.filter(pl.col("timestamp_utc").is_between(start, end))
        if window.height == 0:
            window = sample.head(168)
        fig = px.line(window.to_pandas(), x="timestamp_utc",
                      y=["wind_onshore_actual", "wind_onshore_forecast", "wind_onshore_forecast_intraday"],
                      title="Wind Onshore: Actual vs DA vs Intraday (1-week sample)")
        fig.show()

# Histogram of ratios
ratios = ratio_df.select("ratio").to_pandas()
fig = px.histogram(ratios, x="ratio", nbins=60, title="Ratio Distribution: Intraday / Actual")
fig.show()


In [51]:

# Summary table
summary["Ratio_Stability_Index"] = ratio_stats["cv"]
summary


,DA_MAE,ID_MAE,DA_to_ID_Ratio,Ratio_Stability_Index
0,1102.923208,8187.504576,0.883797,2.239626
